In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

""" 
logits_student : 学生网络的逻辑输出
logits_teacher : 教师网络的逻辑输出
target ：标签值
alpha、beta、temperature : 超参数
"""
def dkd_loss(logits_student, logits_teacher, target, alpha, beta, temperature):
    gt_mask = _get_gt_mask(logits_student, target)
    print('gt_mask', gt_mask)
    other_mask = _get_other_mask(logits_student, target)
    print('other_mask', other_mask)
    pred_student = F.softmax(logits_student / temperature, dim=1)
    pred_teacher = F.softmax(logits_teacher / temperature, dim=1)
    print('pred_student', pred_student)
    print('pred_teacher', pred_teacher)
    pred_student = cat_mask(pred_student, gt_mask, other_mask)
    pred_teacher = cat_mask(pred_teacher, gt_mask, other_mask)
    print('#####pred_student', pred_student)
    print('#####pred_teacher', pred_teacher)
    log_pred_student = torch.log(pred_student)
    print('log_pred_student', log_pred_student)
    tckd_loss = (
        F.kl_div(log_pred_student, pred_teacher, size_average=False)
        * (temperature**2)
        / target.shape[0]
    )
    pred_teacher_part2 = F.softmax(
        logits_teacher / temperature - 1000.0 * gt_mask, dim=1
    )
    print('pred_teacher_part2', pred_teacher_part2)
    log_pred_student_part2 = F.log_softmax(
        logits_student / temperature - 1000.0 * gt_mask, dim=1
    )
    print('log_pred_student_part2', log_pred_student_part2)
    nckd_loss = (
        F.kl_div(log_pred_student_part2, pred_teacher_part2, size_average=False)
        * (temperature**2)
        / target.shape[0]
    )
    return alpha * tckd_loss + beta * nckd_loss

def _get_gt_mask(logits, target):
    target = target.reshape(-1)
    mask = torch.zeros_like(logits).scatter_(1, target.unsqueeze(1), 1).bool()
    return mask


def _get_other_mask(logits, target):
    target = target.reshape(-1)
    mask = torch.ones_like(logits).scatter_(1, target.unsqueeze(1), 0).bool()
    return mask


def cat_mask(t, mask1, mask2):
    t1 = (t * mask1).sum(dim=1, keepdims=True)
    print('t1', t1)
    t2 = (t * mask2).sum(1, keepdims=True)
    print('t2', t2)
    rt = torch.cat([t1, t2], dim=1)
    return rt

In [3]:
# 模拟数据
batch_size = 5
num_classes = 3

# 随机生成学生和教师的logits
logits_student = torch.randn(batch_size, num_classes)
logits_teacher = torch.randn(batch_size, num_classes)
print('student logits:', logits_student)
print('teacher logits:', logits_teacher)

# 生成目标标签，使用 one-hot 编码
target = torch.tensor([0, 1, 2, 0, 2])

# 设置超参数
alpha = 0.5
beta = 0.5
temperature = 2.0

# 计算损失
loss = dkd_loss(logits_student, logits_teacher, target, alpha, beta, temperature)
print("DKD Loss:", loss.item())

student logits: tensor([[-0.8940,  2.0573,  1.6237],
        [ 0.2819,  1.0588,  2.4846],
        [ 0.2918, -0.7683, -0.5047],
        [-0.8088, -0.3582,  0.7157],
        [ 0.7805,  0.2292, -0.2987]])
teacher logits: tensor([[ 1.4825,  1.9460,  0.2149],
        [ 0.2876, -0.2855, -0.5354],
        [ 0.4624, -0.4093,  2.8398],
        [-0.6278,  1.8866, -0.6306],
        [ 1.2145,  0.0937, -0.7354]])
gt_mask tensor([[ True, False, False],
        [False,  True, False],
        [False, False,  True],
        [ True, False, False],
        [False, False,  True]])
other_mask tensor([[False,  True,  True],
        [ True, False,  True],
        [ True,  True, False],
        [False,  True,  True],
        [ True,  True, False]])
pred_student tensor([[0.1124, 0.4917, 0.3959],
        [0.1824, 0.2690, 0.5487],
        [0.4425, 0.2604, 0.2971],
        [0.2275, 0.2850, 0.4875],
        [0.4270, 0.3241, 0.2489]])
pred_teacher tensor([[0.3582, 0.4517, 0.1901],
        [0.4143, 0.3111, 0.2746],


/data2/tongshuo/.local/lib/python3.8/site-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  warnings.warn(warning.format(ret))


In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

""" 
logits_student : 学生网络的逻辑输出
logits_teacher : 教师网络的逻辑输出
target ：标签值
alpha、beta、temperature : 超参数
"""
def dkd_no_labels_loss(logits_student, logits_teacher, alpha, beta, temperature):
    _, target = torch.max(logits_teacher, dim=1)
    gt_mask = _get_gt_mask(logits_student, target)
    print('gt_mask', gt_mask)
    other_mask = _get_other_mask(logits_student, target)
    print('other_mask', other_mask)
    pred_student = F.softmax(logits_student / temperature, dim=1)
    pred_teacher = F.softmax(logits_teacher / temperature, dim=1)
    pred_student = cat_mask(pred_student, gt_mask, other_mask)
    pred_teacher = cat_mask(pred_teacher, gt_mask, other_mask)
    print('pred_student', pred_student)
    print('pred_teacher', pred_teacher)
    log_pred_student = torch.log(pred_student)
    print('log_pred_student', log_pred_student)
    tckd_loss = (
        F.kl_div(log_pred_student, pred_teacher, size_average=False)
        * (temperature**2)
        / target.shape[0]
    )
    pred_teacher_part2 = F.softmax(
        logits_teacher / temperature - 1000.0 * gt_mask, dim=1
    )
    print('pred_teacher_part2', pred_teacher_part2)
    log_pred_student_part2 = F.log_softmax(
        logits_student / temperature - 1000.0 * gt_mask, dim=1
    )
    print('log_pred_student_part2', log_pred_student_part2)
    nckd_loss = (
        F.kl_div(log_pred_student_part2, pred_teacher_part2, size_average=False)
        * (temperature**2)
        / target.shape[0]
    )
    return alpha * tckd_loss + beta * nckd_loss

def _get_gt_mask(logits, target):
    target = target.reshape(-1)
    mask = torch.zeros_like(logits).scatter_(1, target.unsqueeze(1), 1).bool()
    return mask


def _get_other_mask(logits, target):
    target = target.reshape(-1)
    mask = torch.ones_like(logits).scatter_(1, target.unsqueeze(1), 0).bool()
    return mask


def cat_mask(t, mask1, mask2):
    t1 = (t * mask1).sum(dim=1, keepdims=True)
    print('t1', t1)
    t2 = (t * mask2).sum(1, keepdims=True)
    print('t2', t2)
    rt = torch.cat([t1, t2], dim=1)
    return rt

In [18]:
# 模拟数据
batch_size = 5
num_classes = 3

# 随机生成学生和教师的logits
logits_student = torch.randn(batch_size, num_classes)
logits_teacher = torch.randn(batch_size, num_classes)
print('student logits:', logits_student)
print('teacher logits:', logits_teacher)

# 设置超参数
alpha = 0.5
beta = 0.5
temperature = 2.0

# 计算损失
loss = dkd_no_labels_loss(logits_student, logits_teacher, alpha, beta, temperature)
print("DKD Loss:", loss.item())

student logits: tensor([[-2.0045, -0.8501, -1.8528],
        [-0.9082,  0.0936, -0.3802],
        [-0.9519,  0.1837, -1.2764],
        [-0.4722, -2.0396,  1.2616],
        [ 0.1592, -0.6743,  0.2722]])
teacher logits: tensor([[ 1.1481, -1.2348, -0.3069],
        [ 0.3876, -0.7227,  0.2823],
        [ 0.1999, -0.5800, -0.1018],
        [-0.9669,  0.8553,  0.5414],
        [-1.2588, -0.6664, -0.0492]])
gt_mask tensor([[ True, False, False],
        [ True, False, False],
        [ True, False, False],
        [False,  True, False],
        [False, False,  True]])
other_mask tensor([[False,  True,  True],
        [False,  True,  True],
        [False,  True,  True],
        [ True, False,  True],
        [ True,  True, False]])
t1 tensor([[0.2591],
        [0.2530],
        [0.2767],
        [0.1190],
        [0.3894]])
t2 tensor([[0.7409],
        [0.7470],
        [0.7233],
        [0.8810],
        [0.6106]])
t1 tensor([[0.5596],
        [0.3964],
        [0.3941],
        [0.4431],
  

/data2/tongshuo/.local/lib/python3.8/site-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  warnings.warn(warning.format(ret))


In [ ]:
gt_mask = _get_gt_mask(logits_student, target)
other_mask = _get_other_mask(logits_student, target)
pred_student = F.softmax(logits_student / temperature, dim=1)
pred_teacher = F.softmax(logits_teacher / temperature, dim=1)
pred_student = cat_mask(pred_student, gt_mask, other_mask)
pred_teacher = cat_mask(pred_teacher, gt_mask, other_mask)

def cat_mask(t, mask1, mask2):
    t1 = (t * mask1).sum(dim=1, keepdims=True)
    t2 = (t * mask2).sum(1, keepdims=True)
    rt = torch.cat([t1, t2], dim=1)
    return rt